# 14. Diagnóstico de errores del detector

**Qué hace.** Mira dónde se equivoca F, con las predicciones de los sujetos de afuera de la validación anidada (notebook 12). Es descriptivo: no elige ni cambia nada del pipeline. Sirve para el informe y la defensa, y para decidir con datos si vale la pena una decisión en dos niveles (clasificar primero la noche y usar un umbral más exigente en las noches que no son A).

**Preguntas y cortes (fijados antes de correr).**
1. **Falsos positivos** (minutos normales marcados como apnea): qué parte del total y qué tasa hay según el tipo de noche (A, B, C); la distancia al minuto con apnea más cercano (1, 2, de 3 a 5, de 6 a 10, más de 10 minutos, o noche sin apnea; la ventana de 240 s abarca ±2 minutos); la fracción de intervalos corregidos del minuto (0 %, hasta 10 %, de 10 a 20 %); y si la ventana tiene señal perdida.
2. **Falsos negativos** (minutos con apnea no detectados): según el largo del episodio (1 o 2 minutos, de 3 a 5, de 6 a 10, más de 10) y si el minuto está en el borde del episodio o en su interior.
3. **Techo de la decisión en dos niveles:** cuánto subirían la especificidad y cuánto bajaría el sesgo por noche si se eliminaran **todos** los falsos positivos de las noches que no son A, conservando los aciertos. Es un techo optimista: una regla real también perdería algunos aciertos. Se calcula con la categoría real (techo ideal) y con la que estima el detector con la regla de la base (techo realista).
4. **Exactitud lejos de los bordes:** sensibilidad y especificidad sin contar los minutos a 1 o 2 minutos de un cambio de etiqueta.
5. **Sujetos:** área bajo la curva y exactitud dentro de cada sujeto; para los peores, su categoría, índice de apnea-hipopnea, calidad y polaridad del electrocardiograma, y en qué grupo de la validación anidada quedaron.

Nada se decide automáticamente: con esto se discute si la decisión en dos niveles vale la pena o si alcanza con explicar los errores en el informe.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.environ["PYTHONPATH"] = str(ROOT)
from src.config import ECG_FILTER, RESULTS_DIR, SPLIT_CSV
from src.data import load_apnea_labels, load_ecg, load_ecg_digital, load_header, load_qrs, load_record_info
from src.evaluation.errors import error_table, subject_scores
from src.evaluation.metrics import minute_metrics, night_minutes, record_metrics
from src.features.edr import qrs_reference
from src.features.variants import coupling_variants, hrv_variants
from src.preprocessing.filters import apply_zero_phase, design_bandpass
from src.preprocessing.quality import window_signal_quality
from src.stats.compare import subject_folds
from src.stats.nested import nested_cv

FIG = RESULTS_DIR / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
CACHE = RESULTS_DIR / "e3_v3"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8})
AZUL, NARANJA, GRIS, TINTA = "#2a78d6", "#eb6834", "#a3a29c", "#2b2b2b"
split = pd.read_csv(SPLIT_CSV).set_index("record")
DEV = list(split.index[split.set == "dev"])
subject_of = split.loc[DEV, "subject"]

coup = pd.read_parquet(CACHE / "acoplamiento.parquet")
info = coup.groupby(["record", "minute"])[["label", "corrected"]].first()
f_pred, f_ch = CACHE / "anidada_pred.parquet", CACHE / "anidada_elecciones.parquet"
if f_pred.exists() and f_ch.exists():
    pred, choices = pd.read_parquet(f_pred), pd.read_parquet(f_ch)
else:                                    # misma corrida que el notebook 12
    wide_c, meta_c = coupling_variants(coup)
    wide_h, meta_h = hrv_variants(pd.read_parquet(CACHE / "hrv.parquet"))
    eval_idx = (wide_c.dropna().index.intersection(wide_h.dropna().index)
                .intersection(info.index[(info.label >= 0) & (info.corrected <= 0.2)]))
    y_all = info.loc[eval_idx, "label"].to_numpy(int)
    s_all = subject_of.loc[eval_idx.get_level_values("record")].to_numpy()
    folds = subject_folds(pd.Series(y_all, index=s_all).groupby(level=0).max().astype(bool), k=5)
    pred, choices = nested_cv({"F": (wide_c, meta_c), "HRV": (wide_h, meta_h)}, info, subject_of,
                              eval_idx, folds, n_boot=1000, n_boot_rec=300, seed=2026)
del coup
print(f"predicciones de afuera: {len(pred):,} minutos de {pred.index.get_level_values('record').nunique()} noches")

predicciones de afuera: 22,275 minutos de 47 noches


In [2]:
# Etiquetas completas, calidad de la ventana y polaridad de cada noche (sin etiquetas, salvo las del minuto)
labels = {rec: load_apnea_labels(rec) for rec in DEV}
perdida, polaridad = [], {}
for rec in DEV:
    h = load_header(rec)
    d = load_ecg_digital(rec)
    q = window_signal_quality(d, h["fs"], h["bits"], np.arange(len(labels[rec])), width_s=240.0, adc_zero=h["cero"])
    perdida.append(q.assign(record=rec))
    ecg, fs = load_ecg(rec)
    polaridad[rec] = qrs_reference(apply_zero_phase(ecg, design_bandpass(fs, *ECG_FILTER)), load_qrs(rec), fs)[1]
perdida = pd.concat(perdida).set_index(["record", "minute"])

t = error_table(pred, labels)
t["corregidos"] = info.loc[t.index, "corrected"].to_numpy()
t["senal_perdida"] = perdida.loc[t.index, "segundos_perdidos"].fillna(0).to_numpy() > 0
print(t.tipo.value_counts().to_dict())

{'VN': 10044, 'VP': 7426, 'FP': 3377, 'FN': 1428}


## 1. ¿Dónde caen los falsos positivos?

In [3]:
def reparto(df, col, cats):
    # parte del total de errores en cada grupo y tasa de error dentro del grupo
    g = df.groupby(col, observed=False)
    return pd.DataFrame({"minutos": g.size().reindex(cats), "errores": g["error"].sum().reindex(cats),
                         "parte del total": (g["error"].sum() / df["error"].sum()).reindex(cats),
                         "tasa": g["error"].mean().reindex(cats)}).fillna(0)

neg = t[t.label == 0].copy()
neg["error"] = neg.tipo == "FP"
d_ = neg.distancia
neg["distancia_grupo"] = np.select([np.isinf(d_), d_ <= 1, d_ <= 2, d_ <= 5, d_ <= 10],
                                   ["sin apnea en la noche", "1", "2", "3-5", "6-10"], "más de 10")
neg["corregidos_grupo"] = pd.cut(neg.corregidos, [-0.001, 0, 0.1, 0.2], labels=["0 %", "hasta 10 %", "10 a 20 %"])
print(f"falsos positivos: {neg.error.sum():,} de {len(neg):,} minutos normales (especificidad {1 - neg.error.mean():.3f})")
print("\nPor tipo de noche:"); display(reparto(neg, "categoria", list("ABC")).round(3))
print("Por distancia al minuto con apnea más cercano:")
display(reparto(neg, "distancia_grupo", ["1", "2", "3-5", "6-10", "más de 10", "sin apnea en la noche"]).round(3))
print("Por intervalos corregidos del minuto:"); display(reparto(neg, "corregidos_grupo", ["0 %", "hasta 10 %", "10 a 20 %"]).round(3))
print("Por señal perdida en la ventana:"); display(reparto(neg, "senal_perdida", [False, True]).round(3))

falsos positivos: 3,377 de 13,421 minutos normales (especificidad 0.748)

Por tipo de noche:


,minutos,errores,parte del total,tasa
categoria,,,,
A,4788,1796,0.532,0.375
B,3415,1001,0.296,0.293
C,5218,580,0.172,0.111


Por distancia al minuto con apnea más cercano:


,minutos,errores,parte del total,tasa
distancia_grupo,,,,
1,817,591,0.175,0.723
2,578,331,0.098,0.573
3-5,1309,507,0.150,0.387
6-10,1505,452,0.134,0.300
más de 10,7416,1251,0.370,0.169
sin apnea en la noche,1796,245,0.073,0.136


Por intervalos corregidos del minuto:


,minutos,errores,parte del total,tasa
corregidos_grupo,,,,
0 %,10780,2398,0.710,0.222
hasta 10 %,2149,803,0.238,0.374
10 a 20 %,492,176,0.052,0.358


Por señal perdida en la ventana:


,minutos,errores,parte del total,tasa
senal_perdida,,,,
False,13330,3290,0.974,0.247
True,91,87,0.026,0.956


## 2. ¿Dónde caen los falsos negativos?

In [4]:
pos = t[t.label == 1].copy()
pos["error"] = pos.tipo == "FN"
pos["largo_grupo"] = pd.cut(pos.largo_episodio, [0, 2, 5, 10, np.inf], labels=["1-2", "3-5", "6-10", "más de 10"])
pos["posicion"] = np.where(pos.distancia <= 1, "borde", "interior")
print(f"falsos negativos: {pos.error.sum():,} de {len(pos):,} minutos con apnea (sensibilidad {1 - pos.error.mean():.3f})")
print("\nPor largo del episodio (minutos):"); display(reparto(pos, "largo_grupo", ["1-2", "3-5", "6-10", "más de 10"]).round(3))
print("Por posición en el episodio:"); display(reparto(pos, "posicion", ["borde", "interior"]).round(3))
print("Por tipo de noche:"); display(reparto(pos, "categoria", list("ABC")).round(3))

falsos negativos: 1,428 de 8,854 minutos con apnea (sensibilidad 0.839)

Por largo del episodio (minutos):


,minutos,errores,parte del total,tasa
largo_grupo,,,,
1-2,177,42,0.029,0.237
3-5,271,42,0.029,0.155
6-10,471,38,0.027,0.081
más de 10,7935,1306,0.915,0.165


Por posición en el episodio:


,minutos,errores,parte del total,tasa
posicion,,,,
borde,771,143,0.1,0.185
interior,8083,1285,0.9,0.159


Por tipo de noche:


,minutos,errores,parte del total,tasa
categoria,,,,
A,8330,1378,0.965,0.165
B,508,37,0.026,0.073
C,16,13,0.009,0.812


## 3. Techo de la decisión en dos niveles

In [5]:
recs = t.index.get_level_values("record")
nm = night_minutes(recs, t.label, t.pred)
estimada_a = (nm.min_estimados >= 100).reindex(recs).to_numpy()
real_a = (t.categoria == "A").to_numpy()

def techo(no_a):
    p = t.pred.to_numpy().copy()
    p[(t.label.to_numpy() == 0) & no_a] = 0          # se borran todos los falsos positivos de esas noches
    mm = minute_metrics(t.label, p)
    n = night_minutes(recs, t.label, p)
    r = record_metrics(n.min_estimados, n.min_anotados, n.horas, records=n.index)
    return {"especificidad": mm["Esp"], "sensibilidad": mm["Se"], "sesgo por noche (min/h)": r["sesgo"],
            "categoría A/B/C correcta": r["acierto_categoria"]}

tabla = pd.DataFrame({"actual": techo(np.zeros(len(t), bool)),
                      "techo ideal (categoría real)": techo(~real_a),
                      "techo realista (categoría estimada)": techo(~estimada_a)}).T
display(tabla.astype(float).round(3))

,especificidad,sensibilidad,sesgo por noche (min/h),categoría A/B/C correcta
actual,0.748,0.839,4.965,0.574
techo ideal (categoría real),0.866,0.839,0.698,0.957
techo realista (categoría estimada),0.804,0.839,2.797,0.830


## 4. Exactitud lejos de los bordes de los episodios

In [6]:
filas = {}
for k, nombre in ((0, "todos los minutos"), (1, "sin los minutos a 1 de un cambio"), (2, "sin los minutos a 1 o 2 de un cambio")):
    m = t.distancia > k
    mm = minute_metrics(t.label[m], t.pred[m])
    filas[nombre] = {"minutos": int(m.sum()), "sensibilidad": mm["Se"], "especificidad": mm["Esp"],
                     "exactitud balanceada": mm["exactitud_balanceada"]}
display(pd.DataFrame(filas).T.astype(float).round(3))

,minutos,sensibilidad,especificidad,exactitud balanceada
todos los minutos,22275.0,0.839,0.748,0.794
sin los minutos a 1 de un cambio,20687.0,0.841,0.779,0.810
sin los minutos a 1 o 2 de un cambio,19540.0,0.839,0.796,0.817


## 5. ¿Qué sujetos explican la variación entre grupos?

In [7]:
info_rec = load_record_info().set_index("record")
s = subject_scores(pred, subject_of)
grupo = pred.groupby(level="record")["grupo"].first()
rec_de = subject_of.reset_index().groupby("subject")["record"].apply(list)
s["grupo"] = [grupo.loc[rec_de[x][0]] for x in s.index]
s["registros"] = [", ".join(rec_de[x]) for x in s.index]
s["categorías"] = [", ".join(split.loc[rec_de[x], "category"]) for x in s.index]
s["AHI medio"] = [info_rec.loc[rec_de[x], "AHI"].mean() for x in s.index]
s["corregidos medio"] = [t.loc[t.index.get_level_values("record").isin(rec_de[x]), "corregidos"].mean() for x in s.index]
s["con señal perdida"] = [t.loc[t.index.get_level_values("record").isin(rec_de[x]), "senal_perdida"].mean() for x in s.index]
s["polaridad"] = [", ".join("+" if polaridad[r] > 0 else "−" for r in rec_de[x]) for x in s.index]
display(s.sort_values("auc").round(3))
print("Área bajo la curva media de los sujetos de cada grupo:")
display(s.groupby("grupo")["auc"].agg(["mean", "min", "max", "count"]).round(3))

,minutos,apnea,auc,sensibilidad,especificidad,exactitud,grupo,registros,categorías,AHI medio,corregidos medio,con señal perdida,polaridad
subject,,,,,,,,,,,,,
S23,859,0.003,0.566,0.333,0.867,0.865,1,"c10, x18","C, C",0.000,0.003,0.005,"+, +"
S09,980,0.836,0.664,0.387,0.752,0.447,3,"a09, a18","A, A",57.050,0.004,0.000,"+, +"
S18,1226,0.001,0.742,0.000,0.878,0.877,1,"c01, x17, x35","C, C, C",0.000,0.019,0.003,"+, −, +"
S22,1348,0.007,0.758,0.222,0.936,0.931,5,"c06, c07, x34","C, C, C",0.210,0.006,0.000,"+, +, −"
S02,757,0.807,0.782,0.260,0.897,0.383,4,"a02, x14","A, A",74.500,0.034,0.003,"+, +"
S19,942,0.003,0.834,0.000,0.899,0.896,1,"c02, c09","C, C",0.000,0.003,0.010,"−, +"
S04,1019,0.924,0.835,0.884,0.701,0.870,2,"a04, a12","A, A",78.800,0.022,0.000,"+, −"
S05,1971,0.468,0.843,0.953,0.579,0.754,2,"a05, a10, a20, x07","A, A, A, A",31.000,0.003,0.007,"+, −, +, +"
S12,995,0.213,0.863,0.896,0.639,0.693,4,"a17, x12","A, B",33.000,0.010,0.043,"+, +"


Área bajo la curva media de los sujetos de cada grupo:


,mean,min,max,count
grupo,,,,
1,0.753,0.566,0.869,4
2,0.874,0.835,0.915,4
3,0.826,0.664,0.900,4
4,0.841,0.782,0.879,3
5,0.857,0.758,0.932,3


## Conclusión

**Falsos positivos (3.377; especificidad 0,748).**
- **La mayoría está en noches con apnea:** el 53 % cae en noches A, donde el 37,5 % de los minutos normales se marca como apnea. Las noches B aportan el 30 % y las de control el 17 % (tasa del 11 %).
- **Muchos están cerca de los episodios:** el 27 % cae a 1 o 2 minutos de una apnea (tasas del 72 % y del 57 %), y el 55 % a 10 minutos o menos. Es en parte un efecto de la ventana de 240 s, que abarca ±2 minutos, y en parte la oscilación lenta que rodea a los episodios.
- **La calidad pesa:** con algún intervalo corregido en el minuto, la tasa sube del 22 % al 36–37 %. Los 91 minutos con señal perdida en la ventana son falsos positivos casi siempre (96 %), pero son solo el 2,6 % del total. El hueco interpolado parece una oscilación lenta. Es el mismo efecto que evaluó el notebook 10, que no adoptó la exclusión por la regla de no inferioridad; acá se reporta como limitación, sin reabrir esa decisión.

**Falsos negativos (1.428; sensibilidad 0,839).**
- **No son de episodios cortos ni de bordes:** el 92 % está dentro de episodios de más de 10 minutos, y el 90 % en el interior de los episodios.
- **Se concentran en pocos sujetos:** S09 (a09, a18) detecta el 39 % de sus minutos con apnea y S02 (a02, x14), el 26 %. Son sujetos con mucha apnea en los que F queda por debajo del umbral. a02 es la noche con peor calidad de desarrollo (pierde el 41 % de los minutos por la regla del 20 %), y en el notebook 4.1 a09 era una de las noches con más diferencia entre los minutos con apnea y el índice de apnea-hipopnea.

**Variación entre grupos.** El grupo 4 (0,689 de área bajo la curva) tiene a S02 junto con un sujeto de control: el área dentro de cada sujeto es razonable (de 0,78 a 0,88), pero juntarlos baja el área porque F tiene niveles distintos en cada persona. En los sujetos de control el área dentro del sujeto no es informativa, porque tienen entre 0 y 4 minutos con apnea.

**Techo de la decisión en dos niveles** (optimista: borra todos los falsos positivos de las noches que no son A y conserva todos los aciertos):

| | Especificidad | Sesgo por noche (minutos por hora) | Categoría A/B/C correcta |
|---|---|---|---|
| actual | 0,748 | +5,0 | 57 % |
| techo con la categoría real | 0,866 | +0,7 | 96 % |
| techo con la categoría estimada | 0,804 | +2,8 | 83 % |

Con la categoría que el detector puede estimar, el techo es una especificidad 0,056 más alta y la mitad del sesgo por noche. Una regla real ganaría menos, porque también perdería aciertos en las noches B y en las A que se clasifican mal. Además, no cambia el área bajo la curva y tocaría las decisiones por minuto de las que sale la medida por noche de H2.

**Lejos de los bordes.** Sin los minutos a 1 o 2 minutos de un cambio de etiqueta, la especificidad pasa de 0,748 a 0,796 y la sensibilidad queda igual (0,839). Parte de los falsos positivos son de resolución temporal, pero no la mayoría.

**Decisión: no se prueba la decisión en dos niveles.** Con lo que el detector puede estimar, el techo es chico y una regla real ganaría menos; la categoría ya se resuelve con "A contra el resto" (83 %, notebook 13); y cambiaría las decisiones por minuto de una medida principal de H2 con un parámetro elegido después de mirar desarrollo. Los errores que quedan dependen de la persona (S09, S02) y de la calidad de la señal: se documentan en el informe.